In [29]:
# 1. Install DuckDB
!pip install duckdb pandas numpy --quiet

import duckdb
import numpy as np
import pandas as pd
import datetime

print("DuckDB version:", duckdb.__version__)

DuckDB version: 1.3.2


In [30]:
# 2. Generate Synthetic Marketplace Data Pipeline
np.random.seed(42)

N_RIDERS = 8000
N_DRIVERS = 1200
N_REQUESTS = 40000

# --- 1. Dim Riders ---
rider_ids = [f"R_{i:06d}" for i in range(1, N_RIDERS + 1)]
signup_dates = pd.date_range(start="2025-01-01", end="2025-06-30", freq="D")
dim_riders = pd.DataFrame({
    "rider_id": rider_ids,
    "signup_date": np.random.choice(signup_dates, size=N_RIDERS),
    "rider_tier": np.random.choice(["Standard", "Frequent", "VIP"], size=N_RIDERS, p=[0.70, 0.22, 0.08])
})

# --- 2. Dim Drivers ---
driver_ids = [f"D_{i:05d}" for i in range(1, N_DRIVERS + 1)]
dim_drivers = pd.DataFrame({
    "driver_id": driver_ids,
    "onboarding_date": np.random.choice(signup_dates, size=N_DRIVERS),
    "vehicle_type": np.random.choice(["Sedan", "Hatchback", "Auto/Bike"], size=N_DRIVERS, p=[0.35, 0.40, 0.25]),
    "driver_rating": np.round(np.random.uniform(4.2, 5.0, size=N_DRIVERS), 2)
})

# --- 3. Spatial Clusters (Hex Zones) & Hours ---
zones = ["Zone_Downtown_HQ", "Zone_Airport_Hub", "Zone_Tech_Corridor", "Zone_Suburbs_East", "Zone_Residential_West"]
start_time = datetime.datetime(2025, 7, 1, 0, 0, 0)
timestamps = [start_time + datetime.timedelta(minutes=int(m)) for m in np.random.uniform(0, 31 * 24 * 60, N_REQUESTS)]
timestamps.sort()

# Request simulation with peak hour and surge logic
hours = [t.hour for t in timestamps]
is_peak = [(h in [8, 9, 10, 17, 18, 19, 20]) for h in hours]

surge_multipliers = []
for peak in is_peak:
    if peak:
        surge_multipliers.append(np.random.choice([1.0, 1.2, 1.5, 1.8, 2.2], p=[0.20, 0.30, 0.25, 0.15, 0.10]))
    else:
        surge_multipliers.append(np.random.choice([1.0, 1.1, 1.2], p=[0.80, 0.15, 0.05]))

trip_statuses = []
unfulfilled_reasons = []
matched_drivers = []
etas = []
fares = []

for surge, peak in zip(surge_multipliers, is_peak):
    # Higher surge causes slight rider price drop-off; low supply in peak causes unmatched requests
    rand_val = np.random.rand()

    if surge >= 1.8 and rand_val < 0.25:
        trip_statuses.append("Cancelled_Rider")
        unfulfilled_reasons.append("Rider_Surge_Dropoff")
        matched_drivers.append(None)
        etas.append(None)
        fares.append(0.0)
    elif peak and rand_val < 0.18:
        trip_statuses.append("Unmatched")
        unfulfilled_reasons.append("No_Driver_Found")
        matched_drivers.append(None)
        etas.append(None)
        fares.append(0.0)
    elif rand_val < 0.06:
        trip_statuses.append("Cancelled_Driver")
        unfulfilled_reasons.append("Driver_Rejection_Timeout")
        matched_drivers.append(np.random.choice(driver_ids))
        etas.append(np.random.randint(6, 15))
        fares.append(0.0)
    else:
        trip_statuses.append("Completed")
        unfulfilled_reasons.append("None")
        matched_drivers.append(np.random.choice(driver_ids))
        etas.append(np.random.randint(2, 12))
        base_fare = np.random.uniform(8.0, 28.0)
        fares.append(np.round(base_fare * surge, 2))

fact_ride_requests = pd.DataFrame({
    "request_id": [f"REQ_{i:07d}" for i in range(1, N_REQUESTS + 1)],
    "rider_id": np.random.choice(rider_ids, size=N_REQUESTS),
    "driver_id": matched_drivers,
    "pickup_zone": np.random.choice(zones, size=N_REQUESTS, p=[0.35, 0.15, 0.25, 0.15, 0.10]),
    "request_timestamp": timestamps,
    "surge_multiplier": surge_multipliers,
    "eta_pickup_minutes": etas,
    "trip_fare": fares,
    "trip_status": trip_statuses,
    "unfulfilled_reason": unfulfilled_reasons
})

# --- 4. Populate DuckDB Tables ---
con = duckdb.connect("marketplace_analytics.duckdb")

con.execute("CREATE OR REPLACE TABLE dim_riders AS SELECT * FROM dim_riders")
con.execute("CREATE OR REPLACE TABLE dim_drivers AS SELECT * FROM dim_drivers")
con.execute("CREATE OR REPLACE TABLE fact_ride_requests AS SELECT * FROM fact_ride_requests")

print("Tables successfully loaded into DuckDB!")

Tables successfully loaded into DuckDB!


In [31]:
# 3. Verification Query: Inspect Status & Match Rates
con.execute("""
    SELECT
        trip_status,
        unfulfilled_reason,
        COUNT(*) AS request_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct_of_total,
        ROUND(AVG(surge_multiplier), 2) AS avg_surge,
        ROUND(AVG(trip_fare), 2) AS avg_fare
    FROM fact_ride_requests
    GROUP BY 1, 2
    ORDER BY request_count DESC;
""").df()

,trip_status,unfulfilled_reason,request_count,pct_of_total,avg_surge,avg_fare
0,Completed,None,36041,90.10,1.12,20.27
1,Cancelled_Driver,Driver_Rejection_Timeout,1662,4.16,1.02,0.00
2,Unmatched,No_Driver_Found,1542,3.86,1.26,0.00
3,Cancelled_Rider,Rider_Surge_Dropoff,755,1.89,1.96,0.00


In [32]:
# ==============================================================================
# MILESTONE 2: SQL MARKETPLACE LIQUIDITY & DISPATCH LATENCY ENGINE
# ==============================================================================

# Query 1: Zone-Level Marketplace Liquidity & Unfulfilled Attribution
q1_liquidity_by_zone = con.execute("""
    SELECT
        pickup_zone,
        COUNT(*) AS total_demand_pings,
        SUM(CASE WHEN trip_status = 'Completed' THEN 1 ELSE 0 END) AS completed_trips,
        ROUND(SUM(CASE WHEN trip_status = 'Completed' THEN 1.0 ELSE 0 END) * 100.0 / COUNT(*), 2) AS fulfillment_rate_pct,
        SUM(CASE WHEN unfulfilled_reason = 'No_Driver_Found' THEN 1 ELSE 0 END) AS unfulfilled_supply_deficit,
        SUM(CASE WHEN unfulfilled_reason = 'Rider_Surge_Dropoff' THEN 1 ELSE 0 END) AS unfulfilled_surge_dropoff,
        SUM(CASE WHEN unfulfilled_reason = 'Driver_Rejection_Timeout' THEN 1 ELSE 0 END) AS unfulfilled_driver_timeout,
        ROUND(AVG(surge_multiplier), 2) AS avg_surge_multiplier,
        ROUND(SUM(trip_fare), 2) AS total_gross_bookings
    FROM fact_ride_requests
    GROUP BY pickup_zone
    ORDER BY total_demand_pings DESC;
""").df()

print("--- 1. ZONE LIQUIDITY & UNFULFILLED ATTRIBUTION ---")
display(q1_liquidity_by_zone)

--- 1. ZONE LIQUIDITY & UNFULFILLED ATTRIBUTION ---


,pickup_zone,total_demand_pings,completed_trips,fulfillment_rate_pct,unfulfilled_supply_deficit,unfulfilled_surge_dropoff,unfulfilled_driver_timeout,avg_surge_multiplier,total_gross_bookings
0,Zone_Downtown_HQ,13922,12540.0,90.07,545.0,274.0,563.0,1.14,253140.25
1,Zone_Tech_Corridor,10061,9056.0,90.01,367.0,189.0,449.0,1.14,184681.30
2,Zone_Suburbs_East,6075,5469.0,90.02,236.0,125.0,245.0,1.15,110785.31
3,Zone_Airport_Hub,5949,5368.0,90.23,225.0,108.0,248.0,1.14,108312.69
4,Zone_Residential_West,3993,3608.0,90.36,169.0,59.0,157.0,1.14,73478.62


In [33]:
# Query 2: Dispatch Latency & ETA SLA Percentiles (p50, p90, p95)
q2_eta_percentiles = con.execute("""
    SELECT
        pickup_zone,
        COUNT(*) AS completed_trips,
        ROUND(AVG(eta_pickup_minutes), 2) AS avg_pickup_eta_min,
        ROUND(PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY eta_pickup_minutes), 1) AS p50_eta_min,
        ROUND(PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY eta_pickup_minutes), 1) AS p90_eta_min,
        ROUND(PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY eta_pickup_minutes), 1) AS p95_eta_min,
        SUM(CASE WHEN eta_pickup_minutes > 10 THEN 1 ELSE 0 END) AS sla_breaches_gt_10m,
        ROUND(SUM(CASE WHEN eta_pickup_minutes > 10 THEN 1.0 ELSE 0 END) * 100.0 / COUNT(*), 2) AS sla_breach_rate_pct
    FROM fact_ride_requests
    WHERE trip_status = 'Completed'
    GROUP BY pickup_zone
    ORDER BY p95_eta_min DESC;
""").df()

print("--- 2. DISPATCH LATENCY & ETA PERCENTILES ---")
display(q2_eta_percentiles)

--- 2. DISPATCH LATENCY & ETA PERCENTILES ---


,pickup_zone,completed_trips,avg_pickup_eta_min,p50_eta_min,p90_eta_min,p95_eta_min,sla_breaches_gt_10m,sla_breach_rate_pct
0,Zone_Downtown_HQ,12540,6.48,7.0,10.0,11.0,1208.0,9.63
1,Zone_Residential_West,3608,6.52,7.0,11.0,11.0,375.0,10.39
2,Zone_Tech_Corridor,9056,6.53,7.0,10.0,11.0,856.0,9.45
3,Zone_Airport_Hub,5368,6.47,7.0,10.0,11.0,525.0,9.78
4,Zone_Suburbs_East,5469,6.44,6.0,10.0,11.0,537.0,9.82


In [34]:
# Query 3: Hourly Surge Elasticity & Supply Deficit Curve
q3_hourly_surge_curve = con.execute("""
    SELECT
        EXTRACT(HOUR FROM request_timestamp) AS hour_of_day,
        COUNT(*) AS hourly_requests,
        ROUND(AVG(surge_multiplier), 2) AS avg_surge,
        ROUND(SUM(CASE WHEN trip_status = 'Completed' THEN 1.0 ELSE 0 END) * 100.0 / COUNT(*), 2) AS fulfillment_rate_pct,
        ROUND(SUM(CASE WHEN unfulfilled_reason = 'No_Driver_Found' THEN 1.0 ELSE 0 END) * 100.0 / COUNT(*), 2) AS no_driver_rate_pct,
        ROUND(SUM(CASE WHEN unfulfilled_reason = 'Rider_Surge_Dropoff' THEN 1.0 ELSE 0 END) * 100.0 / COUNT(*), 2) AS surge_dropoff_rate_pct,
        ROUND(SUM(trip_fare), 2) AS hourly_gross_revenue
    FROM fact_ride_requests
    GROUP BY 1
    ORDER BY 1 ASC;
""").df()

print("--- 3. HOURLY SURGE & FULFILLMENT MATRIX ---")
display(q3_hourly_surge_curve)

--- 3. HOURLY SURGE & FULFILLMENT MATRIX ---


,hour_of_day,hourly_requests,avg_surge,fulfillment_rate_pct,no_driver_rate_pct,surge_dropoff_rate_pct,hourly_gross_revenue
0,0,1605,1.02,94.64,0.00,0.00,27743.82
1,1,1586,1.02,93.38,0.00,0.00,27401.92
2,2,1651,1.02,94.61,0.00,0.00,29030.47
3,3,1691,1.02,94.50,0.00,0.00,29423.44
4,4,1606,1.03,93.77,0.00,0.00,27847.55
5,5,1691,1.03,94.44,0.00,0.00,29571.99
6,6,1625,1.02,93.97,0.00,0.00,28164.21
7,7,1681,1.03,94.17,0.00,0.00,29011.58
8,8,1686,1.42,79.89,13.58,6.52,34093.84
9,9,1621,1.41,79.58,14.25,6.17,32641.05


In [35]:
# Export aggregated master dataset for downstream modeling and Tableau BI
con.execute("""
    CREATE OR REPLACE TABLE marketplace_master_analytics AS
    SELECT
        r.request_id,
        r.rider_id,
        rd.rider_tier,
        r.driver_id,
        d.vehicle_type,
        d.driver_rating,
        r.pickup_zone,
        r.request_timestamp,
        EXTRACT(HOUR FROM r.request_timestamp) AS request_hour,
        EXTRACT(DOW FROM r.request_timestamp) AS day_of_week,
        CASE WHEN EXTRACT(HOUR FROM r.request_timestamp) IN (8, 9, 10, 17, 18, 19, 20) THEN 'Peak' ELSE 'Off-Peak' END AS period_type,
        r.surge_multiplier,
        r.eta_pickup_minutes,
        r.trip_fare,
        r.trip_status,
        r.unfulfilled_reason
    FROM fact_ride_requests r
    LEFT JOIN dim_riders rd ON r.rider_id = rd.rider_id
    LEFT JOIN dim_drivers d ON r.driver_id = d.driver_id;
""")

# Export to CSV for Tableau
con.execute("COPY marketplace_master_analytics TO 'marketplace_master_analytics.csv' (HEADER, DELIMITER ',');")
print("Master analytics dataset exported: marketplace_master_analytics.csv")

Master analytics dataset exported: marketplace_master_analytics.csv


In [36]:
# ==============================================================================
# MILESTONE 3: PYTHON SURGE ELASTICITY & DEADWEIGHT LOSS ENGINE
# ==============================================================================

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import linregress
from sklearn.linear_model import LogisticRegression

# 1. Fetch data from DuckDB connection into pandas
df = con.execute("SELECT * FROM marketplace_master_analytics").df()

# 2. Surge Multiplier Bucketing & Price Sensitivity Analysis
surge_summary = (
    df.groupby("surge_multiplier")
    .agg(
        total_pings=("request_id", "count"),
        completed_trips=(
            "trip_status",
            lambda x: (x == "Completed").sum(),
        ),
        surge_dropoffs=(
            "unfulfilled_reason",
            lambda x: (x == "Rider_Surge_Dropoff").sum(),
        ),
        no_driver_count=(
            "unfulfilled_reason",
            lambda x: (x == "No_Driver_Found").sum(),
        ),
        total_revenue=("trip_fare", "sum"),
        avg_revenue_per_request=("trip_fare", "mean"),
    )
    .reset_index()
)

surge_summary["fulfillment_rate_pct"] = (
    surge_summary["completed_trips"] / surge_summary["total_pings"]
) * 100
surge_summary["surge_dropoff_rate_pct"] = (
    surge_summary["surge_dropoffs"] / surge_summary["total_pings"]
) * 100
surge_summary["no_driver_rate_pct"] = (
    surge_summary["no_driver_count"] / surge_summary["total_pings"]
) * 100

print("--- 1. SURGE MULTIPLIER PERFORMANCE BREAKDOWN ---")
display(
    surge_summary[
        [
            "surge_multiplier",
            "total_pings",
            "fulfillment_rate_pct",
            "surge_dropoff_rate_pct",
            "no_driver_rate_pct",
            "avg_revenue_per_request",
            "total_revenue",
        ]
    ]
)

# 3. Logistic Regression: Probability of Rider Drop-off vs. Surge Multiplier
df["is_surge_dropoff"] = (
    df["unfulfilled_reason"] == "Rider_Surge_Dropoff"
).astype(int)

X = df[["surge_multiplier"]]
y = df["is_surge_dropoff"]

log_reg = LogisticRegression()
log_reg.fit(X, y)

odds_ratio = np.exp(log_reg.coef_[0][0])
print(
    f"\nSurge Drop-off Odds Ratio per 1.0x Surge Increase: {odds_ratio:.2f}x"
)

# 4. Optimal Surge Cap Optimization
# Identify the multiplier where marginal revenue gain turns negative due to drop-offs
optimal_tier = surge_summary.loc[
    surge_summary["avg_revenue_per_request"].idxmax()
]
print(
    f"Revenue-Maximizing Surge Multiplier: {optimal_tier['surge_multiplier']}x "
    f"(Yielding ${optimal_tier['avg_revenue_per_request']:.2f} avg revenue per demand ping)"
)

--- 1. SURGE MULTIPLIER PERFORMANCE BREAKDOWN ---


,surge_multiplier,total_pings,fulfillment_rate_pct,surge_dropoff_rate_pct,no_driver_rate_pct,avg_revenue_per_request,total_revenue
0,1.0,24971,93.015898,0.000000,1.589844,16.756389,418423.80
1,1.1,4343,94.358738,0.000000,0.000000,18.695690,81195.38
2,1.2,4854,86.341162,0.000000,12.216728,18.608348,90324.92
3,1.5,2944,81.250000,0.000000,18.750000,22.058485,64940.18
4,1.8,1767,73.910583,26.089417,0.000000,24.129909,42637.55
5,2.2,1121,73.773417,26.226583,0.000000,29.327690,32876.34



Surge Drop-off Odds Ratio per 1.0x Surge Increase: 153.73x
Revenue-Maximizing Surge Multiplier: 2.2x (Yielding $29.33 avg revenue per demand ping)
